In [1]:
import MDAnalysis as mda
from numpy import *
import os
from pylab import *
import MDAnalysis.analysis.distances
import MDAnalysis.analysis.rms
from MDAnalysis.analysis import align
import glob
import scipy.stats
import sklearn
import sklearn.decomposition
import sklearn.preprocessing
import pandas as pd
import seaborn as sns

In [2]:
import os
EQPOINT=0

systemFolders = glob.glob("*t5a*/")

systemgros=[]
systemtprs=[]
systemtrjs=[]
for i in range(len(systemFolders)):
    systemgros.append(sorted(glob.glob(systemFolders[i]+"*.gro")))
    systemtprs.append(sorted(glob.glob(systemFolders[i]+"*.tpr")))
    systemtrjs.append(sorted(glob.glob(systemFolders[i]+"*.xtc")))


    
    
threeColor=["#FE6100","#332288","#882255"]
colourScheme= threeColor
system_names = ["rhT5A","T5A","T5AR332P"]
systems=[]
for i in range(len(systemgros)):
    sub=[]
    for j in range(len(systemtprs[i])):
        # When using TPRs, residues are indexed from 1; so we need to add in the first residue, 1 - 1 + first resid=first resid
        firstres = mda.Universe(systemgros[i][j]).residues.resids[0]-1
        tu = mda.Universe(systemtprs[i][j],systemtrjs[i][j])
        tu.residues.resids +=firstres
                          
        sub.append(tu)
        
    systems.append(sub)

proteins=[]
#proteins
for i in range(len(systems)):
    sub=[]
    for j in range(len(systems[i])):
        sub.append(systems[i][j].select_atoms("protein"))
    proteins.append(sub)


bodys=[]
# rhesus v1 is 326-349
# Human v1 is 324-345
huloopstring = "resid 324:345"
rhloopstring = "resid 326:349"
system_loop_strings = [rhloopstring,huloopstring,huloopstring]
for i in range(len(systems)):
    sub = []
    for j in range(len(systems[i])):
        sub.append(systems[i][j].select_atoms("protein and not ("+system_loop_strings[i]+")"))
        
    bodys.append(sub)
    
    
v1s=[]
# rhesus v1 is 326-349
# Human v1 is 324-345
huloopstring = "resid 324:345"
rhloopstring = "resid 326:349"
system_loop_strings = [rhloopstring,huloopstring,huloopstring]
for i in range(len(systems)):
    sub = []
    for j in range(len(systems[i])):
        sub.append(systems[i][j].select_atoms("protein and "+system_loop_strings[i]))
        
    v1s.append(sub)                   
                   

In [3]:
def writeMySel(u,selString,sysname,trjnum,savePath = "./rhNumbering/",resids=arange(498,681)):
    
    
    """ A function that intakes a universe, a selection, and outputs just the selection with renumbered residue IDs"""
    
    #get the trajectorynum
    trjnum=str(trjnum)
    
    sel=u.select_atoms(selString)
    
    sel.residues.resids=resids
    
    sel.write(savePath+sysname+trjnum+"_firstframe.gro")
    
    

    with MDAnalysis.Writer(savePath+sysname+trjnum+".xtc", sel.n_atoms) as W:
        for ts in u.trajectory:
            W.write(sel)


In [16]:
# These are the residues we are going to select to save: there are a couple extra residues (unaligned)_ on either end of eahc protein
# We are just going to exclude those. for simple and mathcing comparison
myselectionList=[]

myselectionList.append("protein and resid 292:495")
myselectionList.append("protein and resid 290:491")
myselectionList.append("protein and resid 290:491")

# We also want to renumber the rhesus resids to the human numbering, and set the inserted V1 residues to their own region (1339 and 1340)
myResidList=[]
myResidList.append(list(arange(290,492)))
myResidList[0].insert(337-290,1339)
myResidList[0].insert(338-290,1340)
myResidList.append(list(arange(290,492)))
myResidList.append(list(arange(290,492)))


for sys in range(len(systems)):
    for trj in range(len(systems[sys])):
        writeMySel(systems[sys][trj],myselectionList[sys],system_names[sys],trj,resids=myResidList[sys])

In [4]:
# I think it would also be useful to have a version where all resids are included as normal,
# but we also exclude the non-matching residues from the ends.
myselectionList=[]

myselectionList.append("protein and resid 292:495")
myselectionList.append("protein and resid 290:491")
myselectionList.append("protein and resid 290:491")

# We also want to renumber the rhesus resids to the human numbering, and set the inserted V1 residues to their own region (1339 and 1340)
myResidList=[]
myResidList.append(list(arange(290,494)))
myResidList.append(list(arange(290,492)))
myResidList.append(list(arange(290,492)))


for sys in range(len(systems)):
    for trj in range(len(systems[sys])):
        writeMySel(systems[sys][trj],myselectionList[sys],system_names[sys],trj,savePath="./naturalNumbering/",resids=myResidList[sys])